In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi']=100
df = pd.read_csv('../Data/Raw/Crime Data.csv')
print('Rows, Columns:', df.shape)
df.head()
df.info()
print('Missing values per column:')
print(df.isna().sum()[df.isna().sum()>0])
print('\nDuplicate rows:', df.duplicated().sum())
df.describe().round(1)
clean = df.copy()
clean = clean.drop_duplicates()
clean['City'] = clean['City'].str.strip().str.title()
clean.loc[(clean['Complainant_Age']<18)|(clean['Complainant_Age']>80),'Complainant_Age'] = np.nan
num_cols = ['Officers_Assigned','Reporting_Delay_Hours','Witness_Count','Complainant_Age','Loss_Amount_INR','Resolution_Days']
for c in num_cols:
    clean[c] = clean[c].fillna(clean[c].median())
clean['Reporting_Mode'] = clean['Reporting_Mode'].fillna(clean['Reporting_Mode'].mode()[0])
print('Rows after cleaning:', len(clean), '| Missing values left:', clean.isna().sum().sum())
clean['Loss_Per_Officer'] = (clean['Loss_Amount_INR']/clean['Officers_Assigned']).round(0)
def status(d):
    if d>75: return 'At Risk'
    if d>60: return 'Delayed'
    if d>40: return 'On Track'
    return 'Fast'
clean['Case_Status'] = clean['Resolution_Days'].apply(status)
clean['Delay_Band'] = pd.cut(clean['Reporting_Delay_Hours'],[0,24,72,120,500],labels=['<24h','1-3 days','3-5 days','>5 days'],right=False)
clean[['Case_ID','Loss_Per_Officer','Case_Status','Delay_Band']].head()

print(clean['Resolution_Days'].describe().round(1))
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.histplot(clean['Resolution_Days'],bins=15,kde=True,color='#1F3864',ax=ax[0]);ax[0].set_title('Distribution of Resolution Days')
order=['Fast','On Track','Delayed','At Risk']
sns.countplot(data=clean,x='Case_Status',hue='Case_Status',legend=False,order=order,palette=['#1F3864','#2E75B6','#9DC3E6','#C00000'],ax=ax[1]);ax[1].set_title('Cases by Case Status')
plt.tight_layout();plt.savefig('../images/01_overall_resolution.png');plt.show()
print(clean.groupby('Delay_Band',observed=True)['Resolution_Days'].agg(['count','mean']).round(1))

fig,ax=plt.subplots(2,2,figsize=(12,8))
for a,col in zip(ax.flat,['City','Crime_Type','Severity','Reporting_Mode']):
    sns.barplot(data=clean,x=col,y='Resolution_Days',errorbar=None,color='#2E75B6',ax=a)
    a.set_title(f'Average Resolution Days by {col}');a.set_ylim(0,100)
    for p in a.patches:a.annotate(f'{p.get_height():.1f}',(p.get_x()+p.get_width()/2,p.get_height()),ha='center',va='bottom',fontsize=9)
plt.tight_layout();plt.savefig('../images/04_group_comparison.png');plt.show()

risk = clean[clean['Case_Status']=='At Risk'].sort_values('Resolution_Days',ascending=False)
print(f'{len(risk)} cases ({len(risk)/len(clean):.1%}) are At Risk')
print('Average reporting delay of At Risk cases:', round(risk['Reporting_Delay_Hours'].mean(),1),'hours vs overall', round(clean['Reporting_Delay_Hours'].mean(),1),'hours')
risk[['Case_ID','City','Crime_Type','Reporting_Delay_Hours','Officers_Assigned','Resolution_Days']].head(10)

